# Smart Greenhouse Operational Lookback Ablation

Controlled model-development experiment: forecast the deployment horizons
`+1h` and `+3h` with one fixed LSTM architecture while changing only the
historical context length (`24h`, `48h`, `72h`). All three experiments use
identical target references. Held-out and final-test partitions remain locked.

## 01 - Operational Forecasting Decision

Notebook 03 established that `+1h` and `+3h` are the operational horizons.
Longer horizons remain research diagnostics and are not optimized here. LSTM
is fixed because it won the predeclared Notebook 03 validation criterion.

## 02 - Configuration

In [ ]:
import os
from pathlib import Path

SEED = 20260816
LOOKBACK_CANDIDATES = (24, 48, 72)
FORECAST_HORIZONS = (1, 3)
MAX_LOOKBACK = max(LOOKBACK_CANDIDATES)
MAX_FORECAST_HORIZON = max(FORECAST_HORIZONS)
BATCH_SIZE = 256
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 7
GRADIENT_CLIP_NORM = 1.0
HIDDEN_SIZE = 64
NUM_LAYERS = 1
NUM_WORKERS = 2

LOOKBACK_ABLATION_SMOKE_TEST = False
LOOKBACK_ABLATION_SMOKE_TEST = os.getenv(
    "GREENHOUSE_LOOKBACK_ABLATION_SMOKE_TEST",
    str(LOOKBACK_ABLATION_SMOKE_TEST),
).lower() in {"1", "true", "yes"}
SMOKE_MAX_EPOCHS = 1
SMOKE_MAX_TRAIN_BATCHES = 3
SMOKE_MAX_VALIDATION_BATCHES = 2

default_root = Path("/content/smart_greenhouse_dataset") if Path("/content").exists() else Path.cwd()
DATA_ROOT = Path(os.getenv("GREENHOUSE_DATA_ROOT", str(default_root))).expanduser()
PREPROCESSING_ARTIFACT_DIR = Path(os.getenv(
    "GREENHOUSE_PREPROCESSING_ARTIFACT_DIR",
    str(DATA_ROOT / "artifacts" / "preprocessing"),
))
LOOKBACK_ABLATION_ARTIFACT_DIR = Path(os.getenv(
    "GREENHOUSE_LOOKBACK_ABLATION_ARTIFACT_DIR",
    str(DATA_ROOT / "artifacts" / (
        "operational_lookback_ablation_smoke"
        if LOOKBACK_ABLATION_SMOKE_TEST
        else "operational_lookback_ablation"
    )),
))
INDEX_FILE = DATA_ROOT / "full_dataset_index.csv"

EXPECTED_SCENARIOS = 24
EXPECTED_ROWS_PER_SCENARIO = 70_128
EXPECTED_TOTAL_ROWS = 1_683_072
EXPECTED_COMMON_TRAIN_PER_SCENARIO = 52_510
EXPECTED_COMMON_VALIDATION_PER_SCENARIO = 8_782
EXPECTED_COMMON_TRAIN_WINDOWS = 1_050_200
EXPECTED_COMMON_VALIDATION_WINDOWS = 175_640
print(f"DATA_ROOT={DATA_ROOT.resolve()}")
print(f"LOOKBACK_ABLATION_SMOKE_TEST={LOOKBACK_ABLATION_SMOKE_TEST}")

## 03 - Imports

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import json
import math
import platform
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler

## 04 - Reproducibility

In [ ]:
def set_reproducibility(seed: int) -> torch.Generator:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


set_reproducibility(SEED)

## 05 - Device Gate

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
elif not LOOKBACK_ABLATION_SMOKE_TEST:
    raise RuntimeError("Full operational lookback ablation requires CUDA")

## 06 - Paths

In [ ]:
def normalize_index_path(raw_path: str) -> Path:
    normalized = str(raw_path).strip().replace("\\", "/")
    if not normalized:
        raise ValueError("Empty indexed path")
    return Path(normalized)


def resolve_scenario_path(raw_path: str, data_root: Path) -> Path:
    normalized = normalize_index_path(raw_path)
    candidates = [
        normalized if normalized.is_absolute() else data_root / normalized,
        data_root / "outputs" / "full_generation" / "ml" / normalized.name,
        data_root / "ml" / normalized.name,
        data_root / normalized.name,
    ]
    unique = list(dict.fromkeys(candidate.resolve() for candidate in candidates))
    existing = [candidate for candidate in unique if candidate.is_file()]
    if unique[0] in existing:
        return unique[0]
    if len(existing) != 1:
        raise FileNotFoundError(f"Cannot uniquely resolve {raw_path!r}: {existing}")
    warnings.warn(f"Explicit path fallback used for {raw_path!r}: {existing[0]}")
    return existing[0]

## 07 - Load Preprocessing Artifacts

In [ ]:
preprocessing_paths = {
    "feature_scaler": PREPROCESSING_ARTIFACT_DIR / "feature_scaler.pkl",
    "target_scaler": PREPROCESSING_ARTIFACT_DIR / "target_scaler.pkl",
    "split_manifest": PREPROCESSING_ARTIFACT_DIR / "split_manifest.json",
    "preprocessing_config": PREPROCESSING_ARTIFACT_DIR / "preprocessing_config.json",
}
missing = [str(path) for path in preprocessing_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Locked preprocessing artifacts missing: {missing}")
feature_scaler = joblib.load(preprocessing_paths["feature_scaler"])
target_scaler = joblib.load(preprocessing_paths["target_scaler"])
split_manifest = json.loads(
    preprocessing_paths["split_manifest"].read_text(encoding="utf-8")
)
preprocessing_config = json.loads(
    preprocessing_paths["preprocessing_config"].read_text(encoding="utf-8")
)
if not LOOKBACK_ABLATION_SMOKE_TEST and preprocessing_config.get("smoke_test_execution"):
    raise ValueError("Full ablation refuses smoke-fitted preprocessing artifacts")
for scaler in (feature_scaler, target_scaler):
    if not all(hasattr(scaler, name) for name in ("mean_", "scale_", "transform")):
        raise TypeError("A fitted StandardScaler-compatible artifact is required")
print(f"Loaded preprocessing artifacts from {PREPROCESSING_ARTIFACT_DIR.resolve()}")

## 08 - Validate Locked Contract

In [ ]:
FEATURE_COLUMNS = [
    "air_temperature", "air_humidity", "soil_temperature", "soil_moisture",
    "light_lux", "pump_state", "fan_state", "grow_light_state",
]
TARGET_COLUMNS = [
    "air_temperature", "air_humidity", "soil_temperature", "soil_moisture", "light_lux",
]
BINARY_FEATURE_COLUMNS = ["pump_state", "fan_state", "grow_light_state"]
SOURCE_COLUMNS = ["timestamp", *FEATURE_COLUMNS]
if preprocessing_config["feature_columns"] != FEATURE_COLUMNS:
    raise ValueError("Locked feature order changed")
if preprocessing_config["target_columns"] != TARGET_COLUMNS:
    raise ValueError("Locked target order changed")
if LOOKBACK_CANDIDATES != (24, 48, 72) or FORECAST_HORIZONS != (1, 3):
    raise ValueError("Operational ablation axes changed")

development_scenario_ids = list(split_manifest["development_scenario_ids"])
held_out_scenario_ids = list(split_manifest["held_out_scenario_ids"])
if len(development_scenario_ids) != 20 or len(held_out_scenario_ids) != 4:
    raise ValueError("Locked split must remain 20 development / 4 held-out")
if set(development_scenario_ids).intersection(held_out_scenario_ids):
    raise ValueError("Development and held-out scenarios overlap")
train_start, train_end = map(pd.Timestamp, split_manifest["train_date_range"])
validation_start, validation_end = map(pd.Timestamp, split_manifest["validation_date_range"])
if (train_start, train_end) != (
    pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2023-12-31 23:00")
):
    raise ValueError("TRAIN range changed")
if (validation_start, validation_end) != (
    pd.Timestamp("2024-01-01 00:00"), pd.Timestamp("2024-12-31 23:00")
):
    raise ValueError("Validation range changed")
print("Locked deployment/split contract PASS")

## 09 - Load Canonical Index

In [ ]:
def load_canonical_index(path: Path) -> pd.DataFrame:
    index = pd.read_csv(path)
    required = {"parameter_set_id", "ml_file", "ml_rows", "config_hash", "validation_status"}
    if required.difference(index.columns):
        raise ValueError("Canonical index schema mismatch")
    if len(index) != EXPECTED_SCENARIOS:
        raise ValueError("Canonical scenario count mismatch")
    if index["parameter_set_id"].duplicated().any() or index["config_hash"].duplicated().any():
        raise ValueError("Duplicate canonical identity/config")
    if not (index["ml_rows"].astype(int) == EXPECTED_ROWS_PER_SCENARIO).all():
        raise ValueError("Canonical row-count mismatch")
    if not (index["validation_status"] == "PASS").all():
        raise ValueError("Non-PASS canonical scenario")
    if set(index["parameter_set_id"]) != set(development_scenario_ids + held_out_scenario_ids):
        raise ValueError("Split IDs do not partition canonical index")
    return index.sort_values("parameter_set_id").reset_index(drop=True)


canonical_index = load_canonical_index(INDEX_FILE)
assert int(canonical_index["ml_rows"].astype(int).sum()) == EXPECTED_TOTAL_ROWS
print(f"Canonical index PASS: {len(canonical_index)} scenarios")

## 10 - Resolve Development Files

In [ ]:
index_by_id = canonical_index.set_index("parameter_set_id")
development_scenario_paths = {
    scenario_id: resolve_scenario_path(index_by_id.loc[scenario_id, "ml_file"], DATA_ROOT)
    for scenario_id in development_scenario_ids
}
if set(development_scenario_paths) != set(development_scenario_ids):
    raise AssertionError("Development path resolution mismatch")
if set(development_scenario_paths).intersection(held_out_scenario_ids):
    raise AssertionError("Held-out path was resolved")
print(f"Resolved development-only files: {len(development_scenario_paths)}")

## 11 - Build Per-Scenario Arrays

In [ ]:
@dataclass(frozen=True)
class ScenarioArrays:
    timestamps: np.ndarray
    scaled_features: np.ndarray
    scaled_targets: np.ndarray
    raw_targets: np.ndarray
    raw_actuators: np.ndarray


def validate_source_frame(frame: pd.DataFrame, scenario_id: str) -> pd.DataFrame:
    if list(frame.columns) != SOURCE_COLUMNS or len(frame) != EXPECTED_ROWS_PER_SCENARIO:
        raise ValueError(f"{scenario_id}: schema/row mismatch")
    frame = frame.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="raise")
    if frame["timestamp"].duplicated().any() or not frame["timestamp"].is_monotonic_increasing:
        raise ValueError(f"{scenario_id}: invalid timestamp order")
    if not (frame["timestamp"].diff().dropna() == pd.Timedelta(hours=1)).all():
        raise ValueError(f"{scenario_id}: non-hourly gap")
    numeric = frame[FEATURE_COLUMNS].to_numpy(np.float64)
    if frame[FEATURE_COLUMNS].isna().any().any() or not np.isfinite(numeric).all():
        raise ValueError(f"{scenario_id}: NaN/Inf")
    for column in BINARY_FEATURE_COLUMNS:
        if not set(frame[column].unique()).issubset({0, 1}):
            raise ValueError(f"{scenario_id}: nonbinary actuator")
    return frame


def frame_to_arrays(frame: pd.DataFrame) -> ScenarioArrays:
    raw_targets = frame[TARGET_COLUMNS].to_numpy(np.float64)
    raw_actuators = frame[BINARY_FEATURE_COLUMNS].to_numpy(np.float32)
    scaled_sensors = feature_scaler.transform(raw_targets).astype(np.float32)
    return ScenarioArrays(
        timestamps=np.ascontiguousarray(frame["timestamp"].to_numpy(dtype="datetime64[ns]")),
        scaled_features=np.ascontiguousarray(np.column_stack([scaled_sensors, raw_actuators])),
        scaled_targets=np.ascontiguousarray(target_scaler.transform(raw_targets).astype(np.float32)),
        raw_targets=np.ascontiguousarray(raw_targets.astype(np.float32)),
        raw_actuators=np.ascontiguousarray(raw_actuators),
    )


active_development_ids = (
    development_scenario_ids[:1]
    if LOOKBACK_ABLATION_SMOKE_TEST
    else development_scenario_ids
)
scenario_arrays: dict[str, ScenarioArrays] = {}
for scenario_id in active_development_ids:
    frame = pd.read_csv(development_scenario_paths[scenario_id])
    scenario_arrays[scenario_id] = frame_to_arrays(validate_source_frame(frame, scenario_id))
print(f"Cached arrays for {len(scenario_arrays)} development scenarios")

## 12 - Common Target Window Semantics

In [ ]:
@dataclass(frozen=True)
class CommonSequenceIndex:
    split_name: str
    scenario_ids: tuple[str, ...]
    scenario_codes: np.ndarray
    input_end_positions: np.ndarray
    target_start: pd.Timestamp
    target_end: pd.Timestamp

    def __len__(self) -> int:
        return int(len(self.input_end_positions))

    def resolve(self, item: int) -> tuple[str, int]:
        return (
            self.scenario_ids[int(self.scenario_codes[item])],
            int(self.input_end_positions[item]),
        )


def common_input_end_positions(
    arrays: ScenarioArrays,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> np.ndarray:
    timestamps = arrays.timestamps
    candidates = np.arange(
        MAX_LOOKBACK - 1,
        len(timestamps) - MAX_FORECAST_HORIZON,
        dtype=np.int64,
    )
    input_start = candidates - MAX_LOOKBACK + 1
    target_matrix = candidates[:, None] + np.asarray(FORECAST_HORIZONS)[None, :]
    targets = timestamps[target_matrix]
    targets_in_split = (
        (targets >= np.datetime64(target_start))
        & (targets <= np.datetime64(target_end))
    ).all(axis=1)
    input_continuous = (
        timestamps[candidates] - timestamps[input_start]
        == np.timedelta64(MAX_LOOKBACK - 1, "h")
    )
    target_reach_continuous = (
        timestamps[candidates + MAX_FORECAST_HORIZON] - timestamps[candidates]
        == np.timedelta64(MAX_FORECAST_HORIZON, "h")
    )
    return candidates[
        targets_in_split & input_continuous & target_reach_continuous
    ].astype(np.int32)


def build_common_sequence_index(
    arrays_by_scenario: dict[str, ScenarioArrays],
    scenario_ids: list[str],
    split_name: str,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> CommonSequenceIndex:
    ordered = tuple(sorted(scenario_ids))
    codes, positions = [], []
    for code_value, scenario_id in enumerate(ordered):
        scenario_positions = common_input_end_positions(
            arrays_by_scenario[scenario_id], target_start, target_end
        )
        codes.append(np.full(len(scenario_positions), code_value, dtype=np.int16))
        positions.append(scenario_positions)
    return CommonSequenceIndex(
        split_name=split_name,
        scenario_ids=ordered,
        scenario_codes=np.concatenate(codes),
        input_end_positions=np.concatenate(positions),
        target_start=target_start,
        target_end=target_end,
    )

## 13 - Build Common Train/Validation Index

In [ ]:
if LOOKBACK_ABLATION_SMOKE_TEST:
    active_train_start = pd.Timestamp("2018-01-01 00:00")
    active_train_end = pd.Timestamp("2018-01-14 23:00")
    active_validation_start = pd.Timestamp("2024-01-01 00:00")
    active_validation_end = pd.Timestamp("2024-01-07 23:00")
else:
    active_train_start, active_train_end = train_start, train_end
    active_validation_start, active_validation_end = validation_start, validation_end

common_train_index = build_common_sequence_index(
    scenario_arrays, active_development_ids, "train", active_train_start, active_train_end
)
common_validation_index = build_common_sequence_index(
    scenario_arrays,
    active_development_ids,
    "validation",
    active_validation_start,
    active_validation_end,
)
actual_common_window_counts = {
    "train": len(common_train_index),
    "validation": len(common_validation_index),
}

train_hours_per_scenario = int((train_end - train_start) / pd.Timedelta(hours=1)) + 1
validation_hours_per_scenario = int(
    (validation_end - validation_start) / pd.Timedelta(hours=1)
) + 1
derived_common_train_per_scenario = (
    train_hours_per_scenario - MAX_LOOKBACK - MAX_FORECAST_HORIZON + 1
)
derived_common_validation_per_scenario = (
    validation_hours_per_scenario - MAX_FORECAST_HORIZON + 1
)
assert derived_common_train_per_scenario == EXPECTED_COMMON_TRAIN_PER_SCENARIO
assert derived_common_validation_per_scenario == EXPECTED_COMMON_VALIDATION_PER_SCENARIO
assert derived_common_train_per_scenario * 20 == EXPECTED_COMMON_TRAIN_WINDOWS
assert derived_common_validation_per_scenario * 20 == EXPECTED_COMMON_VALIDATION_WINDOWS
if not LOOKBACK_ABLATION_SMOKE_TEST and actual_common_window_counts != {
    "train": EXPECTED_COMMON_TRAIN_WINDOWS,
    "validation": EXPECTED_COMMON_VALIDATION_WINDOWS,
}:
    raise ValueError(f"Common window-count mismatch: {actual_common_window_counts}")
print(f"Common train/validation windows: {actual_common_window_counts}")

## 14 - Lookback-Aware Dataset

In [ ]:
class OperationalForecastDataset(Dataset):
    def __init__(
        self,
        arrays: dict[str, ScenarioArrays],
        common_index: CommonSequenceIndex,
        lookback_steps: int,
    ) -> None:
        if lookback_steps not in LOOKBACK_CANDIDATES:
            raise ValueError(f"Unsupported lookback: {lookback_steps}")
        self.arrays = arrays
        self.common_index = common_index
        self.lookback_steps = lookback_steps
        self.horizon_offsets = np.asarray(FORECAST_HORIZONS, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.common_index)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        scenario_id, input_end = self.common_index.resolve(item)
        arrays = self.arrays[scenario_id]
        input_start = input_end - self.lookback_steps + 1
        features = arrays.scaled_features[input_start : input_end + 1]
        targets = arrays.scaled_targets[input_end + self.horizon_offsets]
        if features.shape != (self.lookback_steps, len(FEATURE_COLUMNS)):
            raise RuntimeError("Invalid lookback input shape")
        if targets.shape != (len(FORECAST_HORIZONS), len(TARGET_COLUMNS)):
            raise RuntimeError("Invalid operational target shape")
        return torch.from_numpy(features), torch.from_numpy(targets)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


train_datasets = {
    lookback: OperationalForecastDataset(scenario_arrays, common_train_index, lookback)
    for lookback in LOOKBACK_CANDIDATES
}
validation_datasets = {
    lookback: OperationalForecastDataset(scenario_arrays, common_validation_index, lookback)
    for lookback in LOOKBACK_CANDIDATES
}
effective_workers = 0 if LOOKBACK_ABLATION_SMOKE_TEST else NUM_WORKERS
train_loader_generators = {
    lookback: set_reproducibility(SEED) for lookback in LOOKBACK_CANDIDATES
}
train_loaders = {
    lookback: DataLoader(
        train_datasets[lookback], batch_size=BATCH_SIZE, shuffle=True,
        num_workers=effective_workers, pin_memory=DEVICE.type == "cuda",
        persistent_workers=effective_workers > 0,
        worker_init_fn=seed_worker if effective_workers else None,
        generator=train_loader_generators[lookback],
    )
    for lookback in LOOKBACK_CANDIDATES
}
validation_loaders = {
    lookback: DataLoader(
        validation_datasets[lookback], batch_size=BATCH_SIZE, shuffle=False,
        num_workers=effective_workers, pin_memory=DEVICE.type == "cuda",
        persistent_workers=effective_workers > 0,
        worker_init_fn=seed_worker if effective_workers else None,
    )
    for lookback in LOOKBACK_CANDIDATES
}
assert all(isinstance(loader.sampler, RandomSampler) for loader in train_loaders.values())
assert all(
    isinstance(loader.sampler, SequentialSampler) for loader in validation_loaders.values()
)

## 15 - Shape and Fairness Audit

In [ ]:
sample_batches = {
    lookback: next(iter(train_loaders[lookback]))
    for lookback in LOOKBACK_CANDIDATES
}
for lookback, (features, targets) in sample_batches.items():
    assert features.shape == (BATCH_SIZE, lookback, len(FEATURE_COLUMNS))
    assert targets.shape == (BATCH_SIZE, len(FORECAST_HORIZONS), len(TARGET_COLUMNS))
    assert features.dtype == torch.float32 and targets.dtype == torch.float32
    assert torch.isfinite(features).all() and torch.isfinite(targets).all()

# A controlled ablation reuses exactly one target-reference object.
assert all(dataset.common_index is common_train_index for dataset in train_datasets.values())
assert all(
    dataset.common_index is common_validation_index
    for dataset in validation_datasets.values()
)
reference_item = len(common_train_index) // 2
reference_targets = train_datasets[24][reference_item][1]
for lookback in LOOKBACK_CANDIDATES:
    features, targets = train_datasets[lookback][reference_item]
    torch.testing.assert_close(targets, reference_targets, rtol=0, atol=0)
    torch.testing.assert_close(features[-24:], train_datasets[24][reference_item][0], rtol=0, atol=0)

for index in (common_train_index, common_validation_index):
    for item in (0, len(index) // 2, len(index) - 1):
        scenario_id, input_end = index.resolve(item)
        arrays = scenario_arrays[scenario_id]
        assert input_end - MAX_LOOKBACK + 1 >= 0
        target_positions = input_end + np.asarray(FORECAST_HORIZONS)
        target_timestamps = arrays.timestamps[target_positions]
        assert (target_timestamps >= np.datetime64(index.target_start)).all()
        assert (target_timestamps <= np.datetime64(index.target_end)).all()
boundary_scenario, boundary_input_end = common_validation_index.resolve(0)
boundary_arrays = scenario_arrays[boundary_scenario]
assert boundary_arrays.timestamps[boundary_input_end] < np.datetime64(active_validation_start)
assert boundary_arrays.timestamps[boundary_input_end + 1] == np.datetime64(active_validation_start)
fairness_audit = {
    "same_common_train_index": True,
    "same_common_validation_index": True,
    "same_scenario_codes": True,
    "same_input_end_positions": True,
    "same_target_positions": True,
    "input_shapes": {str(k): list(v[0].shape) for k, v in sample_batches.items()},
    "target_shape": list(reference_targets.shape),
}
print(f"Shape/fairness/leakage PASS: {fairness_audit}")

## 16 - Persistence Baselines

In [ ]:
def baseline_validation_arrays(
    index: CommonSequenceIndex,
    arrays_by_scenario: dict[str, ScenarioArrays],
    max_samples: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    count = len(index) if max_samples is None else min(len(index), max_samples)
    horizons = np.asarray(FORECAST_HORIZONS, dtype=np.int64)
    last_value, daily_seasonal, targets = [], [], []
    for item in range(count):
        scenario_id, input_end = index.resolve(item)
        arrays = arrays_by_scenario[scenario_id]
        last_value.append(np.repeat(arrays.raw_targets[[input_end]], len(horizons), axis=0))
        daily_seasonal.append(arrays.raw_targets[input_end + horizons - 24])
        targets.append(arrays.raw_targets[input_end + horizons])
    return (
        np.asarray(last_value, np.float32),
        np.asarray(daily_seasonal, np.float32),
        np.asarray(targets, np.float32),
    )


max_validation_samples = (
    BATCH_SIZE * SMOKE_MAX_VALIDATION_BATCHES
    if LOOKBACK_ABLATION_SMOKE_TEST
    else None
)
last_value_raw, seasonal_raw, validation_targets_raw = baseline_validation_arrays(
    common_validation_index, scenario_arrays, max_validation_samples
)
assert last_value_raw.shape[1:] == (2, 5)
first_scenario, first_input_end = common_validation_index.resolve(0)
first_arrays = scenario_arrays[first_scenario]
for horizon_index, horizon in enumerate(FORECAST_HORIZONS):
    np.testing.assert_array_equal(
        last_value_raw[0, horizon_index], first_arrays.raw_targets[first_input_end]
    )
    np.testing.assert_array_equal(
        seasonal_raw[0, horizon_index],
        first_arrays.raw_targets[first_input_end + horizon - 24],
    )
    assert first_input_end + horizon - 24 <= first_input_end
print("Operational persistence baselines PASS")

## 17 - LSTM Architecture

In [ ]:
@dataclass(frozen=True)
class OperationalModelConfig:
    input_size: int = 8
    hidden_size: int = 64
    num_layers: int = 1
    num_horizons: int = 2
    output_size: int = 5


class OperationalMultiHorizonLSTM(nn.Module):
    def __init__(self, config: OperationalModelConfig) -> None:
        super().__init__()
        self.config = config
        self.recurrent = nn.LSTM(
            config.input_size,
            config.hidden_size,
            config.num_layers,
            batch_first=True,
        )
        self.output_head = nn.Linear(
            config.hidden_size,
            config.num_horizons * config.output_size,
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.recurrent(features)
        flat = self.output_head(hidden[-1])
        return flat.reshape(-1, self.config.num_horizons, self.config.output_size)


def model_state_hash(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for tensor in model.state_dict().values():
        digest.update(tensor.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


model_config = OperationalModelConfig()
parameter_counts, initialization_hashes = {}, {}
for lookback in LOOKBACK_CANDIDATES:
    set_reproducibility(SEED)
    audit_model = OperationalMultiHorizonLSTM(model_config)
    parameter_counts[lookback] = sum(p.numel() for p in audit_model.parameters())
    initialization_hashes[lookback] = model_state_hash(audit_model)
assert len(set(parameter_counts.values())) == 1
assert len(set(initialization_hashes.values())) == 1
assert (model_config.input_size, model_config.hidden_size, model_config.num_layers) == (8, 64, 1)
print(f"Fixed architecture PASS: parameters={parameter_counts[24]}")

## 18 - Shared Training Utilities

In [ ]:
CHECKPOINT_DIR = LOOKBACK_ABLATION_ARTIFACT_DIR / "checkpoints"
HISTORY_DIR = LOOKBACK_ABLATION_ARTIFACT_DIR / "histories"
METRICS_DIR = LOOKBACK_ABLATION_ARTIFACT_DIR / "metrics"
PLOT_DIR = LOOKBACK_ABLATION_ARTIFACT_DIR / "plots"
for directory in (CHECKPOINT_DIR, HISTORY_DIR, METRICS_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def train_one_epoch(model, loader, optimizer, criterion, max_batches=None) -> dict[str, float]:
    model.train()
    total, count = 0.0, 0
    started = time.perf_counter()
    for batch_index, (features, targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        targets = targets.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        predictions = model(features)
        loss = criterion(predictions, targets)
        if not torch.isfinite(predictions).all() or not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training prediction/loss")
        loss.backward()
        for name, parameter in model.named_parameters():
            if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
                raise FloatingPointError(f"Non-finite gradient: {name}")
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        total += float(loss.detach()) * len(features)
        count += len(features)
    elapsed = time.perf_counter() - started
    if count == 0:
        raise RuntimeError("Training loader produced no samples")
    return {
        "loss": total / count,
        "samples": float(count),
        "duration_seconds": elapsed,
        "samples_per_second": count / elapsed,
    }


@torch.no_grad()
def evaluate_loss(model, loader, criterion, max_batches=None) -> dict[str, float]:
    model.eval()
    total, count = 0.0, 0
    started = time.perf_counter()
    for batch_index, (features, targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        targets = targets.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        predictions = model(features)
        loss = criterion(predictions, targets)
        if not torch.isfinite(predictions).all() or not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation prediction/loss")
        total += float(loss) * len(features)
        count += len(features)
    elapsed = time.perf_counter() - started
    if count == 0:
        raise RuntimeError("Validation loader produced no samples")
    return {"loss": total / count, "samples": float(count), "duration_seconds": elapsed}


def save_checkpoint_atomic(path: Path, payload: dict[str, object]) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def load_checkpoint(path: Path, location):
    try:
        return torch.load(path, map_location=location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=location)


def fit_lookback(lookback_steps: int) -> dict[str, object]:
    set_reproducibility(SEED)
    train_loader_generators[lookback_steps].manual_seed(SEED)
    model = OperationalMultiHorizonLSTM(model_config).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    epochs = SMOKE_MAX_EPOCHS if LOOKBACK_ABLATION_SMOKE_TEST else MAX_EPOCHS
    train_limit = SMOKE_MAX_TRAIN_BATCHES if LOOKBACK_ABLATION_SMOKE_TEST else None
    validation_limit = (
        SMOKE_MAX_VALIDATION_BATCHES if LOOKBACK_ABLATION_SMOKE_TEST else None
    )
    checkpoint_path = CHECKPOINT_DIR / f"best_lstm_lookback{lookback_steps}.pt"
    history, best_loss, best_epoch, patience = [], math.inf, 0, 0
    total_started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        epoch_started = time.perf_counter()
        train_stats = train_one_epoch(
            model, train_loaders[lookback_steps], optimizer, criterion, train_limit
        )
        validation_stats = evaluate_loss(
            model, validation_loaders[lookback_steps], criterion, validation_limit
        )
        epoch_duration = time.perf_counter() - epoch_started
        history.append({
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "validation_loss": validation_stats["loss"],
            "epoch_duration_seconds": epoch_duration,
            "train_samples_per_second": train_stats["samples_per_second"],
            "validation_duration_seconds": validation_stats["duration_seconds"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        })
        if validation_stats["loss"] < best_loss:
            best_loss = validation_stats["loss"]
            best_epoch = epoch
            patience = 0
            save_checkpoint_atomic(checkpoint_path, {
                "model_name": "OperationalMultiHorizonLSTM",
                "model_state_dict": model.state_dict(),
                "model_config": asdict(model_config),
                "lookback_steps": lookback_steps,
                "forecast_horizons": list(FORECAST_HORIZONS),
                "operational_horizons": list(FORECAST_HORIZONS),
                "feature_columns": FEATURE_COLUMNS,
                "target_columns": TARGET_COLUMNS,
                "seed": SEED,
                "epoch": epoch,
                "best_validation_loss": best_loss,
                "strategy": "direct_multi_horizon",
                "future_control_policy": "past_only",
                "loss_definition": "equal-weight MSE over standardized [horizon,target] tensor",
            })
        else:
            patience += 1
        print(
            f"LSTM lookback={lookback_steps} epoch={epoch:02d} "
            f"train={train_stats['loss']:.6f} val={validation_stats['loss']:.6f}"
        )
        if patience >= EARLY_STOPPING_PATIENCE:
            break
    training_duration = time.perf_counter() - total_started
    checkpoint = load_checkpoint(checkpoint_path, DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_validation_loss": float(best_loss),
        "training_duration_seconds": training_duration,
        "checkpoint_path": checkpoint_path,
        "parameter_count": sum(p.numel() for p in model.parameters()),
    }


experiment_results: dict[int, dict[str, object]] = {}

## 19 - Train Lookback 24h

In [ ]:
experiment_results[24] = fit_lookback(24)

## 20 - Train Lookback 48h

In [ ]:
experiment_results[48] = fit_lookback(48)

## 21 - Train Lookback 72h

In [ ]:
experiment_results[72] = fit_lookback(72)
assert len({result["parameter_count"] for result in experiment_results.values()}) == 1

## 22 - Validation Predictions

In [ ]:
def transform_target_cube(raw_cube: np.ndarray) -> np.ndarray:
    shape = raw_cube.shape
    transformed = target_scaler.transform(
        raw_cube.reshape(-1, len(TARGET_COLUMNS)).astype(np.float64)
    )
    return transformed.reshape(shape).astype(np.float32)


@torch.no_grad()
def predict_validation(lookback_steps: int) -> tuple[np.ndarray, np.ndarray, float]:
    model = experiment_results[lookback_steps]["model"]
    model.eval()
    predictions, targets = [], []
    max_batches = (
        SMOKE_MAX_VALIDATION_BATCHES if LOOKBACK_ABLATION_SMOKE_TEST else None
    )
    started = time.perf_counter()
    for batch_index, (features, batch_targets) in enumerate(
        validation_loaders[lookback_steps]
    ):
        if max_batches is not None and batch_index >= max_batches:
            break
        predictions.append(model(features.to(DEVICE)).cpu().numpy())
        targets.append(batch_targets.numpy())
    elapsed = time.perf_counter() - started
    return np.concatenate(predictions), np.concatenate(targets), elapsed


validation_predictions_scaled = {}
validation_targets_scaled_by_lookback = {}
validation_inference_seconds = {}
for lookback in LOOKBACK_CANDIDATES:
    predictions, targets, duration = predict_validation(lookback)
    validation_predictions_scaled[lookback] = predictions
    validation_targets_scaled_by_lookback[lookback] = targets
    validation_inference_seconds[lookback] = duration
true_validation_scaled = validation_targets_scaled_by_lookback[24]
for lookback in LOOKBACK_CANDIDATES:
    np.testing.assert_allclose(
        validation_targets_scaled_by_lookback[lookback],
        true_validation_scaled,
        rtol=0,
        atol=0,
    )
    assert validation_predictions_scaled[lookback].shape[1:] == (2, 5)
    assert np.isfinite(validation_predictions_scaled[lookback]).all()
baseline_target_scaled = transform_target_cube(validation_targets_raw)
np.testing.assert_allclose(true_validation_scaled, baseline_target_scaled, rtol=2e-5, atol=2e-5)
last_value_scaled = transform_target_cube(last_value_raw)
seasonal_scaled = transform_target_cube(seasonal_raw)
print("Identical validation target references PASS")

## 23 - Metrics by Lookback and Horizon

In [ ]:
def inverse_target_cube(scaled_cube: np.ndarray) -> np.ndarray:
    shape = scaled_cube.shape
    return target_scaler.inverse_transform(
        scaled_cube.reshape(-1, len(TARGET_COLUMNS))
    ).reshape(shape)


def metrics_by_horizon(
    lookback_steps: int,
    prediction_scaled: np.ndarray,
    target_scaled: np.ndarray,
) -> list[dict[str, object]]:
    prediction_raw = inverse_target_cube(prediction_scaled)
    target_raw = inverse_target_cube(target_scaled)
    rows = []
    for horizon_index, horizon in enumerate(FORECAST_HORIZONS):
        row = {
            "LookbackHours": lookback_steps,
            "HorizonHours": horizon,
            "standardized_MSE": float(np.mean(
                (prediction_scaled[:, horizon_index] - target_scaled[:, horizon_index]) ** 2
            )),
        }
        for target_index, target_name in enumerate(TARGET_COLUMNS):
            truth = target_raw[:, horizon_index, target_index].astype(np.float64)
            prediction = prediction_raw[:, horizon_index, target_index].astype(np.float64)
            residual = truth - prediction
            denominator = float(np.sum((truth - truth.mean()) ** 2))
            r2 = (
                0.0
                if denominator <= np.finfo(np.float64).eps
                else 1.0 - float(np.sum(residual**2)) / denominator
            )
            row[f"{target_name}_MAE"] = float(np.mean(np.abs(residual)))
            row[f"{target_name}_RMSE"] = float(np.sqrt(np.mean(residual**2)))
            row[f"{target_name}_R2"] = r2
        rows.append(row)
    return rows


lookback_metric_rows = [
    row
    for lookback in LOOKBACK_CANDIDATES
    for row in metrics_by_horizon(
        lookback, validation_predictions_scaled[lookback], true_validation_scaled
    )
]
lookback_validation_by_horizon = pd.DataFrame(lookback_metric_rows)
if len(lookback_validation_by_horizon) != 6:
    raise AssertionError("Expected 3 lookbacks x 2 horizons")
if not np.isfinite(
    lookback_validation_by_horizon.select_dtypes(include=[np.number]).to_numpy()
).all():
    raise FloatingPointError("Lookback validation metrics contain NaN/Inf")

baseline_metrics = {
    "LastValuePersistence": metrics_by_horizon(
        0, last_value_scaled, true_validation_scaled
    ),
    "DailySeasonalPersistence": metrics_by_horizon(
        0, seasonal_scaled, true_validation_scaled
    ),
}
print("Physical metrics by lookback/horizon PASS")

## 24 - Per-Target Lookback Analysis

In [ ]:
target_comparison_rows = []
for _, row in lookback_validation_by_horizon.iterrows():
    for target_name in TARGET_COLUMNS:
        target_comparison_rows.append({
            "LookbackHours": int(row["LookbackHours"]),
            "HorizonHours": int(row["HorizonHours"]),
            "Target": target_name,
            "MAE": float(row[f"{target_name}_MAE"]),
            "RMSE": float(row[f"{target_name}_RMSE"]),
            "R2": float(row[f"{target_name}_R2"]),
        })
lookback_target_comparison = pd.DataFrame(target_comparison_rows)
assert len(lookback_target_comparison) == 3 * 2 * 5
assert np.isfinite(
    lookback_target_comparison.select_dtypes(include=[np.number]).to_numpy()
).all()

## 25 - Accuracy Improvement Analysis

In [ ]:
aggregate_validation_mse = {
    lookback: float(np.mean(
        (validation_predictions_scaled[lookback] - true_validation_scaled) ** 2
    ))
    for lookback in LOOKBACK_CANDIDATES
}


def relative_mse_improvement(reference_mse: float, candidate_mse: float) -> float:
    if reference_mse <= 0:
        raise ValueError("Reference MSE must be positive")
    return (reference_mse - candidate_mse) / reference_mse


relative_improvements = {
    "48_vs_24": relative_mse_improvement(
        aggregate_validation_mse[24], aggregate_validation_mse[48]
    ),
    "72_vs_24": relative_mse_improvement(
        aggregate_validation_mse[24], aggregate_validation_mse[72]
    ),
    "72_vs_48": relative_mse_improvement(
        aggregate_validation_mse[48], aggregate_validation_mse[72]
    ),
}
BEST_ACCURACY_LOOKBACK = min(
    aggregate_validation_mse, key=aggregate_validation_mse.get
)
assert np.isfinite(list(relative_improvements.values())).all()

## 26 - Runtime / Efficiency Analysis

In [ ]:
summary_rows = []
for lookback in LOOKBACK_CANDIDATES:
    result = experiment_results[lookback]
    history = result["history"]
    input_elements = lookback * len(FEATURE_COLUMNS)
    input_bytes_per_batch = BATCH_SIZE * input_elements * np.dtype(np.float32).itemsize
    summary_rows.append({
        "LookbackHours": lookback,
        "BestEpoch": int(result["best_epoch"]),
        "BestValidationMSE": float(result["best_validation_loss"]),
        "MeasuredValidationMSE": aggregate_validation_mse[lookback],
        "TrainDurationSeconds": float(result["training_duration_seconds"]),
        "MeanEpochSeconds": float(np.mean([
            record["epoch_duration_seconds"] for record in history
        ])),
        "ValidationInferenceSeconds": float(validation_inference_seconds[lookback]),
        "MeanTrainSamplesPerSecond": float(np.mean([
            record["train_samples_per_second"] for record in history
        ])),
        "ParameterCount": int(result["parameter_count"]),
        "InputElementsPerSample": input_elements,
        "ApproxInputBytesPerBatch": int(input_bytes_per_batch),
        "RelativeMSEImprovementVs24h": (
            0.0
            if lookback == 24
            else relative_mse_improvement(
                aggregate_validation_mse[24], aggregate_validation_mse[lookback]
            )
        ),
    })
lookback_summary = pd.DataFrame(summary_rows)
assert len(lookback_summary) == 3
assert lookback_summary["ParameterCount"].nunique() == 1
assert lookback_summary["InputElementsPerSample"].tolist() == [192, 384, 576]
assert np.isfinite(lookback_summary.select_dtypes(include=[np.number]).to_numpy()).all()

## 27 - Practical Trade-off Summary

In [ ]:
practicality_rows = []
reference_row = lookback_summary.set_index("LookbackHours").loc[24]
for _, row in lookback_summary.iterrows():
    practicality_rows.append({
        "LookbackHours": int(row["LookbackHours"]),
        "AggregateValidationMSE": float(row["MeasuredValidationMSE"]),
        "AccuracyImprovementVs24h": float(row["RelativeMSEImprovementVs24h"]),
        "SequenceLengthMultiplierVs24h": float(row["LookbackHours"] / 24),
        "EpochTimeMultiplierVs24h": float(
            row["MeanEpochSeconds"] / reference_row["MeanEpochSeconds"]
        ),
        "InputMemoryMultiplierVs24h": float(
            row["ApproxInputBytesPerBatch"]
            / reference_row["ApproxInputBytesPerBatch"]
        ),
    })
practical_tradeoff = pd.DataFrame(practicality_rows)
practical_analysis = {
    "best_accuracy_lookback": int(BEST_ACCURACY_LOOKBACK),
    "selection_policy": "report accuracy/cost trade-off without an arbitrary gain threshold",
    "scientific_status": (
        "SMOKE_INTEGRATION_ONLY"
        if LOOKBACK_ABLATION_SMOKE_TEST
        else "FULL_VALIDATION_ABLATION_AVAILABLE"
    ),
    "relative_improvements": relative_improvements,
}
assert np.isfinite(practical_tradeoff.select_dtypes(include=[np.number]).to_numpy()).all()
print(f"Practicality analysis ready: {practical_analysis['scientific_status']}")

## 28 - Diagnostic Plots

In [ ]:
def plot_loss(history, lookback, path):
    epochs = [record["epoch"] for record in history]
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [record["train_loss"] for record in history], label="Train")
    plt.plot(epochs, [record["validation_loss"] for record in history], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Standardized MSE")
    plt.title(f"LSTM lookback {lookback}h")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


for lookback in LOOKBACK_CANDIDATES:
    plot_loss(
        experiment_results[lookback]["history"],
        lookback,
        PLOT_DIR / f"loss_{lookback}h.png",
    )

plt.figure(figsize=(7, 4))
plt.plot(
    list(LOOKBACK_CANDIDATES),
    [aggregate_validation_mse[value] for value in LOOKBACK_CANDIDATES],
    marker="o",
)
plt.xlabel("Lookback (hours)")
plt.ylabel("Aggregate validation standardized MSE")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PLOT_DIR / "validation_mse_vs_lookback.png", dpi=150)
plt.close()

for horizon in FORECAST_HORIZONS:
    figure, axes = plt.subplots(3, 2, figsize=(11, 11))
    rows = lookback_validation_by_horizon[
        lookback_validation_by_horizon["HorizonHours"] == horizon
    ]
    for axis, target_name in zip(axes.flat, TARGET_COLUMNS):
        axis.plot(rows["LookbackHours"], rows[f"{target_name}_MAE"], marker="o")
        axis.set_title(target_name)
        axis.set_xlabel("Lookback (hours)")
        axis.set_ylabel("MAE")
        axis.grid(alpha=0.2)
    axes.flat[-1].axis("off")
    figure.suptitle(f"Physical MAE at +{horizon}h")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f"mae_{horizon}h_vs_lookback.png", dpi=150)
    plt.close(figure)

plt.figure(figsize=(7, 5))
plt.scatter(
    lookback_summary["MeanEpochSeconds"],
    lookback_summary["BestValidationMSE"],
)
for _, row in lookback_summary.iterrows():
    plt.annotate(
        f"{int(row['LookbackHours'])}h",
        (row["MeanEpochSeconds"], row["BestValidationMSE"]),
    )
plt.xlabel("Mean epoch time (seconds)")
plt.ylabel("Best validation standardized MSE")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PLOT_DIR / "accuracy_runtime_tradeoff.png", dpi=150)
plt.close()

## 29 - Save Artifacts

In [ ]:
def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


for lookback in LOOKBACK_CANDIDATES:
    write_json(
        HISTORY_DIR / f"lookback{lookback}_history.json",
        experiment_results[lookback]["history"],
    )
lookback_summary.to_csv(METRICS_DIR / "lookback_summary.csv", index=False)
lookback_validation_by_horizon.to_csv(
    METRICS_DIR / "lookback_validation_by_horizon.csv", index=False
)
lookback_target_comparison.to_csv(
    METRICS_DIR / "lookback_target_comparison.csv", index=False
)
practical_tradeoff.to_csv(METRICS_DIR / "practical_tradeoff.csv", index=False)
write_json(METRICS_DIR / "persistence_baselines.json", baseline_metrics)
write_json(METRICS_DIR / "practical_analysis.json", practical_analysis)
lookback_ablation_manifest = {
    "seed": SEED,
    "strategy": "direct_multi_horizon",
    "experiment_axis": "lookback_length_only",
    "lookback_candidates": list(LOOKBACK_CANDIDATES),
    "forecast_horizons": list(FORECAST_HORIZONS),
    "operational_horizons": list(FORECAST_HORIZONS),
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "future_control_policy": "past_only",
    "model_config": asdict(model_config),
    "parameter_count": parameter_counts[24],
    "common_target_index": True,
    "expected_full_window_counts": {
        "train": EXPECTED_COMMON_TRAIN_WINDOWS,
        "validation": EXPECTED_COMMON_VALIDATION_WINDOWS,
    },
    "actual_execution_window_counts": actual_common_window_counts,
    "development_scenario_ids": development_scenario_ids,
    "held_out_scenario_ids_provenance_only": held_out_scenario_ids,
    "held_out_csv_loaded": False,
    "final_test_loaders_constructed": False,
    "final_tests_executed": False,
    "lookback_ablation_smoke_test": LOOKBACK_ABLATION_SMOKE_TEST,
}
write_json(
    LOOKBACK_ABLATION_ARTIFACT_DIR / "lookback_ablation_manifest.json",
    lookback_ablation_manifest,
)
print(f"Artifacts saved to {LOOKBACK_ABLATION_ARTIFACT_DIR.resolve()}")

## 30 - Checkpoint Reload Verification

In [ ]:
@torch.no_grad()
def verify_reload(lookback_steps: int) -> dict[str, object]:
    features = next(iter(validation_loaders[lookback_steps]))[0].to(DEVICE)
    trained_model = experiment_results[lookback_steps]["model"].eval()
    reference = trained_model(features).cpu()
    path = CHECKPOINT_DIR / f"best_lstm_lookback{lookback_steps}.pt"
    checkpoint = load_checkpoint(path, "cpu")
    if checkpoint["lookback_steps"] != lookback_steps:
        raise ValueError("Checkpoint lookback mismatch")
    if checkpoint["forecast_horizons"] != list(FORECAST_HORIZONS):
        raise ValueError("Checkpoint horizon mismatch")
    if checkpoint["strategy"] != "direct_multi_horizon":
        raise ValueError("Checkpoint strategy mismatch")
    fresh = OperationalMultiHorizonLSTM(
        OperationalModelConfig(**checkpoint["model_config"])
    )
    fresh.load_state_dict(checkpoint["model_state_dict"])
    fresh = fresh.to(DEVICE).eval()
    reloaded = fresh(features).cpu()
    assert features.shape[1:] == (lookback_steps, 8)
    assert reloaded.shape[1:] == (2, 5)
    torch.testing.assert_close(reference, reloaded, rtol=1e-6, atol=1e-7)
    return {
        "status": "PASS",
        "input_shape": list(features.shape),
        "output_shape": list(reloaded.shape),
        "checkpoint": str(path),
    }


checkpoint_reload_audit = {
    lookback: verify_reload(lookback) for lookback in LOOKBACK_CANDIDATES
}
print(f"Checkpoint reload PASS: {checkpoint_reload_audit}")

## 31 - Final Summary

In [ ]:
experiment_summary = {
    "status": "PASS",
    "experiment": "operational_lookback_ablation",
    "lookback_candidates": list(LOOKBACK_CANDIDATES),
    "forecast_horizons": list(FORECAST_HORIZONS),
    "input_shapes": {str(value): ["B", value, 8] for value in LOOKBACK_CANDIDATES},
    "target_shape": ["B", 2, 5],
    "common_target_index": True,
    "expected_full_window_counts": {
        "train": EXPECTED_COMMON_TRAIN_WINDOWS,
        "validation": EXPECTED_COMMON_VALIDATION_WINDOWS,
    },
    "actual_execution_window_counts": actual_common_window_counts,
    "development_scenarios": len(development_scenario_ids),
    "held_out_scenarios": len(held_out_scenario_ids),
    "same_target_references": True,
    "same_scenario_references": True,
    "same_scalers": True,
    "same_architecture": True,
    "same_parameter_count": True,
    "same_seed_policy": True,
    "scaler_refit_performed": False,
    "held_out_csv_loaded": False,
    "final_test_loaders_constructed": False,
    "final_tests_executed": False,
    "checkpoint_reload": checkpoint_reload_audit,
    "physical_metrics": "PASS",
    "efficiency_summary": "PASS",
    "lookback_ablation_smoke_test": LOOKBACK_ABLATION_SMOKE_TEST,
    "full_lookback_ablation_executed": not LOOKBACK_ABLATION_SMOKE_TEST,
    "scientific_conclusion_status": practical_analysis["scientific_status"],
}
print(json.dumps(experiment_summary, indent=2))